# Profile Before You Trust It

## Project Overview
A new upstream source lands as a flat file and someone asks to "just load it into the warehouse." This notebook is the gate that file has to pass first: an automated data profile that surfaces missingness, duplicates, format drift, and outliers **before** ingestion — so data quality rules are written from evidence, not guesses.

The example uses a raw merchant-contact extract (the kind of CRM drop every data team has seen) and `fg-data-profiling` — the maintained successor of `pandas-profiling` / `ydata-profiling` — to generate a full interactive report with one line of code.

> **Data note:** the dataset is synthetic and messy *on purpose* — every defect was injected by `data/generate_sample_data.py` (missing values, duplicate merchants, four phone formats, lost ZIP leading zeros, sentinel dates, volume outliers). All names and contacts are fabricated.

In [1]:
import pandas as pd

# pandas-profiling -> ydata-profiling (2023) -> fg-data-profiling (2026)
from data_profiling import ProfileReport

### Step 1: First contact with the file

Before any tooling: how big is it, what does it look like, what did pandas guess for types?

In [2]:
df = pd.read_csv('data/merchant_contacts_raw.csv')  # synthetic sample - see data/generate_sample_data.py
df.head(10)

,record_id,company_name,contact_first,contact_last,email,phone,city,state,zip,signup_date,monthly_volume,status
0,MRC-100643,Summit Landscaping,Karen,Wilson,karen.wilson@summit-example.com,4678091155,Phoenix,AZ,85001.0,2024-04-24,-854.36,pending
1,MRC-101972,Ironwood Cleaning,Karen,Miller,karen.miller@ironwood-example.com,(682) 685-3869,Portland,OR,97201.0,2023-06-15,437.15,churned
2,MRC-100581,Bluebird Catering,Paul,Rodriguez,paul.rodriguez@bluebird-example.com,9494065168,Denver,CO,80202.0,2023-08-09,869.11,churned
3,MRC-100051,Summit Logistics,Daniel,Davis,daniel.davis@summit-example.com,(772) 761-4807,Newark,NJ,7102.0,2024-06-21,4801.89,active
4,MRC-100366,Harbor Logistics,Paul,Garcia,NaN,NaN,Phoenix,AZ,85001.0,2019-02-24,1263.92,active
5,MRC-100300,Prairie Logistics,Karen,Brown,karen.brown@prairie-example.com,NaN,Denver,CO,80202.0,2021-02-24,1123.37,CHURNED
6,MRC-101947,Ironwood Printing,Kevin,Davis,kevin.davis@ironwood-example.com,7945137062,Newark,NJ,7102.0,2021-12-25,1986.71,churned
7,MRC-101139,Copper HVAC,Olivia,Thomas,olivia.thomas@copper-example.com,275-663-4287,Spring,tx,NaN,2023-07-18,345.22,active
8,MRC-100509,Pioneer Auto Repair,Robert,Miller,robert.miller@pioneer-example.com,(244) 888-3616,Phoenix,AZ,85001.0,2024-11-21,1202.76,Active
9,MRC-101163,Copper Plumbing,Michael,Hall,michael.hall@copper-example.com,(416) 524-1520,Denver,co,80202.0,2020-04-16,8448.72,active


In [3]:
print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.dtypes

shape: 2,060 rows x 12 columns


record_id          object
company_name       object
contact_first      object
contact_last       object
email              object
phone              object
city               object
state              object
zip               float64
signup_date        object
monthly_volume    float64
status             object
dtype: object

Two smells already: `zip` parsed as a *float-ish* mixed column instead of a 5-character string (leading zeros are gone for some New England rows), and `signup_date` came in as plain `object`.

### Step 2: Targeted checks

The two questions worth asking any contact list up front — what's missing, and who is in here twice?

In [4]:
missing = (
    df.isna().mean()
      .sort_values(ascending=False)
      .to_frame('missing_rate')
      .query('missing_rate > 0')
)
missing.style.format({'missing_rate': '{:.1%}'})

,missing_rate
phone,11.5%
contact_first,6.7%
zip,5.0%
email,3.4%
signup_date,2.3%


In [5]:
exact_dupes = df.duplicated(subset=[col for col in df.columns if col != 'record_id']).sum()
print(f"exact duplicate rows (ignoring record_id): {exact_dupes}")

# fuzzy duplicates: same person + company once case is normalized
key = (
    df['company_name'].str.lower().fillna('')
    + '|' + df['contact_last'].str.lower().fillna('')
    + '|' + df['city'].str.lower().fillna('')
)
fuzzy_groups = key.value_counts()
print(f"case-insensitive merchant keys appearing more than once: {(fuzzy_groups > 1).sum()}")

exact duplicate rows (ignoring record_id): 12
case-insensitive merchant keys appearing more than once: 124


### Step 3: The one-liner profile

`ProfileReport` computes per-column distributions, missingness patterns, duplicate detection, correlations, and cross-column alerts — and writes a self-contained interactive HTML report that can be shared with anyone, no Python required.

In [6]:
profile = ProfileReport(
    df,
    title='Merchant Contacts - Raw Extract Profile',
    dataset={
        'description': 'Synthetic merchant-contact extract profiled before warehouse ingestion.',
        'url': 'https://github.com/dymytryo/notebooks/tree/main/data_profiling',
    },
)
profile.to_file('profile_report.html')

In [7]:
alerts = profile.description_set.alerts
print(f"alerts raised: {len(alerts)}\n")
for a in alerts:
    print(f"- {a}")

alerts raised: 10

- Dataset has 12 (0.6%) duplicate rows
- [city] is highly overall correlated with [state] and 1 other fields
- [state] is highly overall correlated with [city] and 1 other fields
- [zip] is highly overall correlated with [city] and 1 other fields
- [contact_first] 139 (6.7%) missing values
- [email] 71 (3.4%) missing values
- [phone] 237 (11.5%) missing values
- [zip] 104 (5.0%) missing values
- [signup_date] 47 (2.3%) missing values
- [monthly_volume] is highly skewed(γ1 = 45.38721974026523)


### What the profile caught

Reading the report (and the alert list above) against the known defects:

1. **Missingness**, led by `phone` at ~11% — a completeness rule and a "phone required for call campaigns" flag follow directly.
2. **Duplicate rows** — both exact copies and case-variant merchants (`SUMMIT PLUMBING` vs `Summit Plumbing`), so entity resolution is needed before this joins any dimension.
3. **`zip` type drift** — mixed 4- and 5-digit values betray a lost leading zero; store as text, zero-pad on ingest.
4. **`monthly_volume` outliers** — negatives (refunds leaked into a revenue field), zeros, and a single \$1.2B row that would wreck any aggregate it touches.
5. **Categorical drift** — `status` carries `active` / `Active` / `ACTIVE` as three distinct values; `state` mixes case and spelled-out variants.
6. **Date sentinels** — `1900-01-01` placeholders and future signup dates that a freshness test should reject.

## Conclusions

- **Evidence, not guesses:** every ingestion rule above (completeness thresholds, dedup keys, type casts, accepted ranges, value sets) came straight out of one profiling pass.
- **Cheap to automate:** the same report can run on every new file drop in CI and be diffed against the previous profile to catch drift.
- **Shareable:** the HTML report gives non-Python stakeholders the full picture — the negotiation about what "clean enough" means happens around an artifact, not a hunch.